# 🧬 Remedia — Modal GPU Formu

Bu notebook'ta kod satırı değiştirmene gerek yok.

1. Modal'da **GPU = L4**, **CPU = 4**, **RAM = 8 GiB** seç.
2. Kalıcı dosya istiyorsan `remedia-data` Volume'unu `/mnt/remedia-data` yoluna bağla.
3. Aşağıdaki hücreyi bir kez çalıştır.
4. Formu doldur ve **Remedia'yı Başlat** düğmesine bas.

İş bitince kernel'i **Terminate** et.


In [ ]:
# Bu hücreyi bir kez çalıştır; ardından aşağıdaki formu kullan.
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.request
import zipfile
from pathlib import Path

if importlib.util.find_spec("ipywidgets") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "ipywidgets>=8.1.0"],
        check=True,
    )

import ipywidgets as widgets
from IPython.display import FileLink, display

PERSISTENT_ROOT = "/mnt/remedia-data"
REPO_REF = "main"

preset = widgets.Dropdown(
    options=[
        ("Kendim UniProt ID gireceğim", "custom"),
        ("Carbonic anhydrase II — P00918", "P00918"),
        ("EGFR — P00533", "P00533"),
        ("ACE2 — Q9BYF1", "Q9BYF1"),
        ("Estrogen receptor alpha — P03372", "P03372"),
    ],
    value="custom",
    description="Reseptör:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="100%"),
)
uniprot = widgets.Text(
    value="P00918",
    placeholder="Örn. P00533",
    description="UniProt ID:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="100%"),
)
method = widgets.Dropdown(
    options=[
        ("Fusion — önerilen", "fusion"),
        ("Genetic", "genetic"),
        ("BRICS", "brics"),
        ("Random", "random"),
        ("Pretrained / REINVENT4", "pretrained"),
    ],
    value="fusion",
    description="Yöntem:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="100%"),
)
generate_n = widgets.IntSlider(
    value=20, min=5, max=100, step=5,
    description="Molekül sayısı:",
    style={"description_width": "120px"},
    continuous_update=False,
    layout=widgets.Layout(width="100%"),
)
profile = widgets.Dropdown(
    options=[("Balanced — hızlı", "balanced"), ("Final — daha yavaş", "final")],
    value="balanced",
    description="Doğruluk:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="100%"),
)
docking_mode = widgets.Dropdown(
    options=[
        ("İki aşamalı — önerilen", "iki_asamali"),
        ("Sadece fast", "sadece_fast"),
        ("Sadece accurate", "sadece_accurate"),
    ],
    value="iki_asamali",
    description="Docking:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="100%"),
)
box_dim = widgets.IntSlider(
    value=20, min=10, max=40, step=2,
    description="Kutu boyutu:",
    style={"description_width": "120px"},
    continuous_update=False,
    layout=widgets.Layout(width="100%"),
)
top_fraction = widgets.FloatSlider(
    value=0.10, min=0.05, max=0.50, step=0.05,
    readout_format=".2f",
    description="Accurate oranı:",
    style={"description_width": "120px"},
    continuous_update=False,
    layout=widgets.Layout(width="100%"),
)
ga_generations = widgets.IntSlider(
    value=3, min=1, max=10, step=1,
    description="GA nesli:",
    style={"description_width": "120px"},
    continuous_update=False,
    layout=widgets.Layout(width="100%"),
)
run_benchmark = widgets.Checkbox(value=False, description="Ek hız benchmark'ı çalıştır")
force_refresh = widgets.Checkbox(value=False, description="Pocket cache'i yenile")
install_reinvent = widgets.Checkbox(value=False, description="REINVENT4 kur")

advanced = widgets.VBox([
    docking_mode,
    box_dim,
    top_fraction,
    ga_generations,
    run_benchmark,
    force_refresh,
    install_reinvent,
])
accordion = widgets.Accordion(children=[advanced])
accordion.set_title(0, "Gelişmiş ayarlar")

run_button = widgets.Button(
    description="Remedia'yı Başlat",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="100%", height="48px"),
)
output = widgets.Output(layout=widgets.Layout(border="1px solid #555", padding="8px"))
status = widgets.HTML(value="<b>Durum:</b> Formu doldur, sonra yeşil düğmeye bas.")


def _preset_changed(change):
    if change["new"] != "custom":
        uniprot.value = change["new"]


preset.observe(_preset_changed, names="value")


def _install_python_packages():
    required = {
        "requests": "requests>=2.31.0",
        "rdkit": "rdkit>=2023.9.1",
        "Bio": "biopython>=1.81",
        "pandas": "pandas>=2.0.0",
        "numpy": "numpy>=1.24.0",
        "scipy": "scipy>=1.11.0",
        "meeko": "meeko>=0.5.0",
        "gemmi": "gemmi>=0.6.5",
        "openbabel": "openbabel-wheel>=3.1.1",
        "tqdm": "tqdm>=4.66.0",
        "yaml": "pyyaml>=6.0",
        "PIL": "pillow>=10.0.0",
    }
    missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
    if missing:
        print("📦 Eksik Python paketleri kuruluyor...")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)


def _workspace():
    persistent = Path(PERSISTENT_ROOT)
    if persistent.exists():
        return persistent.resolve(), True
    fallback = Path("/root/remedia_workspace").resolve()
    fallback.mkdir(parents=True, exist_ok=True)
    print("⚠️ Volume bağlı değil; kernel kapanınca sonuçlar silinebilir.")
    return fallback, False


def _install_repo(workspace):
    repo_dir = workspace / "Remedia"
    if (repo_dir / "src").is_dir():
        return repo_dir
    image_repo = Path("/opt/remedia")
    if (image_repo / "src").is_dir():
        print("⚡ Remedia özel Modal imajından kopyalanıyor...")
        shutil.copytree(image_repo, repo_dir)
        return repo_dir
    print("📥 Remedia indiriliyor...")
    url = f"https://github.com/mehmetg06/Remedia/archive/refs/heads/{REPO_REF}.zip"
    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp)
        archive_path = tmp_path / "remedia.zip"
        urllib.request.urlretrieve(url, archive_path)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(tmp_path)
        extracted = next(tmp_path.glob("Remedia-*"))
        shutil.copytree(extracted, repo_dir)
    return repo_dir


def _install_tools(workspace):
    tools = workspace / ".remedia-tools"
    (tools / "bin").mkdir(parents=True, exist_ok=True)

    gnina = Path("/usr/local/bin/gnina")
    if not gnina.exists():
        gnina = tools / "bin" / "gnina"
        if not gnina.exists():
            print("📦 GNINA kuruluyor...")
            urllib.request.urlretrieve(
                "https://github.com/gnina/gnina/releases/download/v1.3/gnina",
                gnina,
            )
            gnina.chmod(0o755)

    fpocket = Path("/opt/remedia-fpocket/bin/fpocket")
    if not fpocket.exists():
        fpocket = tools / "fpocket" / "bin" / "fpocket"
        if not fpocket.exists():
            micromamba = tools / "bin" / "micromamba"
            if not micromamba.exists():
                print("📦 micromamba kuruluyor...")
                with tempfile.TemporaryDirectory() as tmp:
                    archive = Path(tmp) / "micromamba.tar.bz2"
                    urllib.request.urlretrieve(
                        "https://micro.mamba.pm/api/micromamba/linux-64/latest",
                        archive,
                    )
                    with tarfile.open(archive, "r:bz2") as tf:
                        member = tf.getmember("bin/micromamba")
                        member.name = "micromamba"
                        tf.extract(member, tools / "bin")
                micromamba.chmod(0o755)
            print("📦 fpocket kuruluyor...")
            subprocess.run([
                str(micromamba), "create", "-y",
                "-p", str(tools / "fpocket"),
                "-c", "conda-forge", "-c", "bioconda", "fpocket",
            ], check=True)
    return gnina, fpocket, tools


def _run_pipeline(settings):
    import datetime
    import pandas as pd

    _install_python_packages()
    workspace, persistent = _workspace()
    repo_dir = _install_repo(workspace)
    gnina_path, fpocket_path, tools_dir = _install_tools(workspace)

    src_dir = repo_dir / "src"
    if str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))
    os.environ["REMEDIA_HOME"] = str(repo_dir)
    os.environ["REMEDIA_WORKSPACE"] = str(workspace)
    os.environ["GNINA_PATH"] = str(gnina_path)
    os.environ["PATH"] = (
        str(fpocket_path.parent) + os.pathsep + str(tools_dir / "bin")
        + os.pathsep + os.environ.get("PATH", "")
    )

    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if gpu.returncode != 0:
        raise RuntimeError("GPU görünmüyor. Compute ekranından L4 seç ve kernel'i yeniden başlat.")

    cache_dir = workspace / "remedia_cache"
    results_root = workspace / "Remedia_results"
    cache_dir.mkdir(parents=True, exist_ok=True)
    results_root.mkdir(parents=True, exist_ok=True)
    run_id = datetime.datetime.now().strftime("run_%Y%m%d_%H%M%S")
    results_dir = results_root / run_id
    results_dir.mkdir(parents=True, exist_ok=True)

    from progress import ProgressReporter
    reporter = ProgressReporter(results_dir / "00_REMEDIA_REPORT")
    reporter.stage("receptor", task="AlphaFold yapısı indiriliyor")

    print("✅ Ortam hazır")
    print(f"🎯 UniProt: {settings['uniprot_id']}")
    print(f"📁 Sonuçlar: {results_dir}")
    print(f"💾 Kalıcı Volume: {'evet' if persistent else 'hayır'}")

    import fetch_structure
    import pocket_detection
    from known_ligands import fetch_known_ligands
    from rdkit import Chem

    cache_key = settings["uniprot_id"].strip().upper()
    receptor = str(fetch_structure.fetch_alphafold(cache_key))
    pocket_cache_path = cache_dir / "pocket_cache.json"
    try:
        pocket_cache = json.loads(pocket_cache_path.read_text()) if pocket_cache_path.exists() else {}
    except Exception:
        pocket_cache = {}

    reporter.stage("pocket", task="Bağlanma cebi belirleniyor")
    cached = None if settings["force_refresh"] else pocket_cache.get(cache_key)
    if cached:
        center = tuple(float(x) for x in cached["center"])
        print(f"⚡ Pocket cache kullanıldı: {center}")
    else:
        pocket = pocket_detection.best_druggable_pocket(Path(receptor))
        center = tuple(round(float(x), 3) for x in pocket["center"])
        pocket_cache[cache_key] = {
            "center": list(center),
            "source": "fpocket",
            "druggability": pocket.get("druggability"),
        }
        pocket_cache_path.write_text(json.dumps(pocket_cache, indent=2))
        print(f"✅ Pocket merkezi: {center}")

    reporter.stage("seeds", task="Bilinen ligandlar getiriliyor")
    ligands, message = fetch_known_ligands(cache_key, max_results=5)
    print(message)
    seeds = [item["smiles"] for item in ligands]
    if not seeds:
        seeds = ["NS(=O)(=O)c1ccccc1", "CC(=O)Nc1nnc(s1)S(N)(=O)=O"]
    seeds = [smiles for smiles in seeds if Chem.MolFromSmiles(smiles)]
    if not seeds:
        raise RuntimeError("Geçerli tohum molekül bulunamadı.")

    from molecule_generator import (
        brics_recombination,
        fusion_generation,
        genetic_algorithm,
        random_mutation,
        write_smi,
    )

    method_value = settings["method"]
    n = settings["generate_n"]
    reporter.stage("generate", total=n, task="Moleküller üretiliyor")
    if method_value == "pretrained":
        if not settings["install_reinvent"]:
            raise RuntimeError("Pretrained için 'REINVENT4 kur' kutusunu işaretle.")
        # Phase 3+: REINVENT4 runs behind the BaseGenerator interface. The
        # default generator is reinvent4, so this makes the same underlying
        # install_reinvent + generate_with_reinvent calls as before.
        from generators import build_generator
        generator = build_generator(settings.get("generator", "reinvent4"), log_fn=print)
        generation_result = generator.generate(
            target=cache_key,
            n=n,
            seeds=seeds,
            output_path=results_dir / "generated.smi",
            cache_dir=cache_dir,
            reporter=reporter,
        )
        generated = generation_result.smiles
    elif method_value == "fusion":
        final, _ = fusion_generation(
            seeds,
            docking_opts=None,
            log_fn=print,
            population_size=max(10, n),
            generations=settings["ga_generations"],
        )
        generated = [smiles for smiles, _score in final]
    elif method_value == "genetic":
        final, _ = genetic_algorithm(
            seeds,
            generations=settings["ga_generations"],
            population_size=max(10, n),
            docking_opts=None,
            log_fn=print,
        )
        generated = [smiles for smiles, _score in final]
    elif method_value == "brics":
        generated = brics_recombination(seeds, n=n)
    else:
        generated = random_mutation(seeds, n=n)

    if method_value != "pretrained" and len(generated) < n:
        pool = set(generated)
        pool.update(random_mutation(seeds, n=n))
        pool.update(brics_recombination(seeds, n=n))
        generated = list(pool)
    generated = [s for s in generated if s][:n]
    if not generated:
        raise RuntimeError("Molekül üretilemedi.")

    reporter.update(len(generated), total=len(generated),
                    message=f"Üretildi: {len(generated)} molekül")
    molecules = [(f"mol_{i:03d}", smiles) for i, smiles in enumerate(generated, 1)]
    # Faz 1: canlı deney konsolu için üretilen molekülleri tek tek bildir.
    try:
        import live_ranking
        live_ranking.emit_generated(reporter, molecules)
    except Exception:
        live_ranking = None
    if method_value != "pretrained":
        write_smi(generated, results_dir / "generated.smi", prefix="mol")

    reporter.stage("dock_fast", total=len(molecules), task="Docking / poz tahmini")
    import gnina_engine  # still used by the optional benchmark below
    # Phase 5: pose prediction runs behind the BasePosePredictor interface.
    # Default engine is GNINA, which returns the exact same rows as before.
    from pose import build_pose_predictor, PoseUnavailable, GninaPredictor
    box_size = (float(settings["box_dim"]),) * 3
    dock_out = results_dir / "gnina_out"
    pose_engine_name = settings.get("pose_engine", "gnina")
    _pose_kwargs = dict(
        profile=settings["profile"],
        docking_mode=settings["docking_mode"],
        top_fraction=settings["top_fraction"],
        gnina_path=str(gnina_path),
        diffdock_results_csv=results_dir / "diffdock_results.csv",
        log_fn=print,
    )
    predictor = build_pose_predictor(pose_engine_name, **_pose_kwargs)
    try:
        pose_result = predictor.predict_pose(
            molecules, receptor=receptor, center=center, size=box_size,
            out_dir=dock_out, reporter=reporter,
        )
    except PoseUnavailable as exc:
        print(f"Poz motoru '{pose_engine_name}' kullanılamadı ({exc}); GNINA'ya dönülüyor.")
        predictor = GninaPredictor(
            profile=settings["profile"], docking_mode=settings["docking_mode"],
            top_fraction=settings["top_fraction"], gnina_path=str(gnina_path), log_fn=print,
        )
        pose_result = predictor.predict_pose(
            molecules, receptor=receptor, center=center, size=box_size,
            out_dir=dock_out, reporter=reporter,
        )
    rows, stage_info = pose_result.rows, pose_result.stage_info

    reporter.update(len(molecules), total=len(molecules),
                    message="GNINA docking tamamlandı")
    docking_df = pd.DataFrame(rows)
    display(docking_df)
    print("✅ GNINA süreç sayısı:", stage_info.get("gnina_processes"))

    if settings["run_benchmark"]:
        bench_rows, summary = gnina_engine.benchmark_fast_vs_accurate(
            molecules[:3],
            receptor=receptor,
            center=center,
            size=(float(settings["box_dim"]),) * 3,
            gnina_path=str(gnina_path),
            out_dir=results_dir / "benchmark",
            profile=settings["profile"],
        )
        pd.DataFrame(bench_rows).to_csv(results_dir / "benchmark.csv", index=False)
        print(summary)

    reporter.stage("admet", total=len(molecules), task="ADMET filtreleme")
    import admet_filter
    admet_df = pd.DataFrame([
        admet_filter.lipinski_veber_filter(smiles, name)
        for name, smiles in molecules
    ])
    docking_csv = results_dir / "docking_scores.csv"
    admet_csv = results_dir / "admet_results.csv"
    ranked_csv = results_dir / "final_ranking.csv"
    reporter.stage("rank", task="Adaylar sıralanıyor")
    docking_df.to_csv(docking_csv, index=False)
    admet_df.to_csv(admet_csv, index=False)
    subprocess.run([
        sys.executable,
        str(src_dir / "rank_report.py"),
        "--docking", str(docking_csv),
        "--admet", str(admet_csv),
        "--output", str(ranked_csv),
    ], check=True)

    ranked_df = pd.read_csv(ranked_csv)
    display(ranked_df.head(10))

    # Phase 6: composite Remedia Score. Additive — the docking-only
    # final_ranking.csv above is kept as the fallback ranking.
    try:
        import remedia_score
        smiles_by_name = {name: smi for name, smi in molecules}
        admet_by_name = {r.get("ligand"): r for r in admet_df.to_dict("records")}
        composite_candidates = []
        for row in docking_df.to_dict("records"):
            name = row.get("ligand")
            cand = dict(row)
            cand["smiles"] = smiles_by_name.get(name, "")
            a = admet_by_name.get(name, {})
            for key in ("MW", "LogP", "TPSA", "HBD", "HBA", "RotB", "pass", "violations"):
                if key in a:
                    cand[key] = a[key]
            cand["admet_pass"] = a.get("pass")
            composite_candidates.append(cand)
        remedia_ranked = (
            live_ranking.score_and_stream(composite_candidates, reporter=reporter)
            if live_ranking is not None
            else remedia_score.rank_candidates(composite_candidates)
        )
        remedia_csv = results_dir / "remedia_ranking.csv"
        remedia_score.write_ranking_csv(remedia_ranked, remedia_csv)
        top = remedia_ranked[0]
        print(f"🏅 Geçici Heuristik Skor (v0) sıralaması: {remedia_csv}")
        print(f"   En iyi aday: {top.get('ligand')} · Heuristik Skor (v0)={top.get('remedia_score')}")
    except Exception as exc:
        print(f"Remedia Score hesaplanamadı; docking sıralaması kullanılacak: {exc}")

    reporter.stage("report", task="Sonuç paketi hazırlanıyor")
    archive = shutil.make_archive(str(results_dir), "zip", root_dir=results_dir)
    print("🎉 Tamamlandı")
    print(f"📄 En önemli dosya: {ranked_csv}")
    display(FileLink(archive))


def _start(_):
    target = uniprot.value.strip().upper()
    if not re.fullmatch(r"[A-Z0-9]{6,10}", target):
        status.value = "<b style='color:#ff6b6b'>Geçerli bir UniProt ID gir. Örn. P00533</b>"
        return

    settings = {
        "uniprot_id": target,
        "method": method.value,
        "generate_n": int(generate_n.value),
        "profile": profile.value,
        "docking_mode": docking_mode.value,
        "box_dim": int(box_dim.value),
        "top_fraction": float(top_fraction.value),
        "ga_generations": int(ga_generations.value),
        "run_benchmark": bool(run_benchmark.value),
        "force_refresh": bool(force_refresh.value),
        "install_reinvent": bool(install_reinvent.value),
    }

    run_button.disabled = True
    status.value = "<b>Durum:</b> Çalışıyor. Bu sayfayı kapatma."
    with output:
        output.clear_output()
        try:
            _run_pipeline(settings)
            status.value = "<b style='color:#45d483'>Durum: Tamamlandı ✅</b>"
        except Exception as exc:
            status.value = "<b style='color:#ff6b6b'>Durum: Hata oluştu ❌</b>"
            print(f"HATA: {type(exc).__name__}: {exc}")
            raise
        finally:
            run_button.disabled = False


run_button.on_click(_start)

form = widgets.VBox([
    widgets.HTML("<h3>🧬 Remedia ayarları</h3><p>Kutuları doldur; kod değiştirme.</p>"),
    preset,
    uniprot,
    method,
    generate_n,
    profile,
    accordion,
    run_button,
    status,
    output,
])
display(form)
